In [1]:
%pip install -e git+https://github.com/saumya06/crewAI_nlsql.git@nlsql#egg=crewai
%cd '/home/wsuser/work/src/crewai'
%pip install src
%cd '/home/wsuser/work/'
%pip install crewai[tools] 

Obtaining crewai from git+https://github.com/saumya06/crewAI_nlsql.git@nlsql#egg=crewai
  Cloning https://github.com/saumya06/crewAI_nlsql.git (to revision nlsql) to ./src/crewai
  Running command git clone --filter=blob:none --quiet https://github.com/saumya06/crewAI_nlsql.git /home/wsuser/work/src/crewai
  Running command git checkout -b nlsql --track origin/nlsql
  Switched to a new branch 'nlsql'
  branch 'nlsql' set up to track 'origin/nlsql'.
  Resolved https://github.com/saumya06/crewAI_nlsql.git to commit f0855c76257f5a048f223f237495da7ef389465c
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━

/opt/conda/envs/Python-RT24.1/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [70 lines of output]
      /opt/conda/envs/Python-RT24.1/lib/python3.11/site-packages/setuptools/_distutils/dist.py:268: UserWarning: Unknown distribution option: 'tests_require'
        warnings.warn(msg)
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib
      creating build/lib/src
      copying src/__init__.py -> build/lib/src
      running egg_info
      writing src.egg-info/PKG-INFO
      writing dependency_links to src.egg-info/dependency_links.txt
      deleting src.egg-info/entry_points.txt
      writing requirements to src.egg-info/requires.txt
      writing top-level names to src.egg-info/top_level.txt
      reading manifest file 'src.egg-info/SOURCES.txt'
      reading manifest template 'MANIFEST.in'
      adding license file 'LICENSE.rst'
      writi

In [1]:
from crewai_tools import NL2SQLTool
from crewai import LLM
import os
from crewai import Agent, Crew, Task, Process

WATSONX_API_KEY = "YOUR_APIKEY"
IBM_CLOUD_URL = "https://us-south.ml.cloud.ibm.com"
WATSONX_MODEL_ID = "watsonx/meta-llama/llama-3-3-70b-instruct"
POSTGRESURL = "POSTGRES_URL"

/opt/conda/envs/Python-RT24.1/lib/python3.11/site-packages/pydantic/_internal/_config.py:291: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.8/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)
/opt/conda/envs/Python-RT24.1/lib/python3.11/site-packages/pydantic/_internal/_fields.py:201: UserWarning: Field name "schema" in "DatabricksQueryToolSchema" shadows an attribute in parent "BaseModel"
  warnings.warn(
/opt/conda/envs/Python-RT24.1/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:404: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.


In [2]:
nl2sql = NL2SQLTool(db_uri=POSTGRESURL)

In [3]:

# Initialize LLM
llm_db = LLM(
    api_key = WATSONX_API_KEY,
    api_base = IBM_CLOUD_URL,
    model = WATSONX_MODEL_ID,
    params={
        "decoding_method": "greedy",
        "max_new_tokens": 15000,
        "temperature": 0,
        "repetition_penalty": 1.05
    }
)

# Initialize LLM
llm_ba = LLM(
    api_key = WATSONX_API_KEY,
    api_base = IBM_CLOUD_URL,
    model = WATSONX_MODEL_ID,
    params={
        "decoding_method": "sample",
        "max_new_tokens": 15000,
        "temperature": 0.8,
        "repetition_penalty": 0.5
    }
)

def create_dbagent() -> Agent:
    return Agent(
        role="Database Analyst",
        goal="""Generate the report based on user question and information in database. Always write the column name as table_name."column_name" in sql query.""",
        backstory="""You are an expert data analyst creating reports from database.""",
        llm=llm_db,
        allow_delegation=False,
        tools=[nl2sql]
    )

def create_researcher() -> Agent:
    return Agent(
        role="Business Analyst",
        goal="""Analyse the data and user question to form the report. If asked about sales, get and analyze information such as monthly quantity, value and percentage chage, peak month, average shipping times.""",
        backstory="""You are an expert data analyst creating reports from database.""",
        llm=llm_ba,
        allow_delegation=True,
    )

def create_generation_task(agent, natural_query):
    return Task(
        description=f"""{natural_query}.""",
        expected_output="""Response from database after running the query.""",
        agent=agent
    )

def create_analysis_task(agent, natural_query):
    return Task(
        description=f"""{natural_query}.""",
        expected_output="""Table or textual analysis report from data. Reply in Thai.""",
        agent=agent
    )

In [4]:
research_agent=create_researcher()
query_agent=create_dbagent()
# generation_task = create_generation_task(research_agent, "how have the sales of Xtralife changed over the past year")
# user_query = "how have the sales of Xtralife product changed over the past year"
user_query = "ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา"

analysis_task = create_analysis_task(research_agent, user_query)
generation_crew = Crew(
    agents=[research_agent, query_agent],
    tasks=[analysis_task],
    verbose=True
    )

In [5]:
crew_output = generation_crew.kickoff()

# Accessing the crew output
print(f"Raw Output: {crew_output.raw}")
if crew_output.json_dict:
    print(f"JSON Output: {json.dumps(crew_output.json_dict, indent=2)}")
if crew_output.pydantic:
    print(f"Pydantic Output: {crew_output.pydantic}")
print(f"Tasks Output: {crew_output.tasks_output}")
print(f"Token Usage: {crew_output.token_usage}")

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6a004c8c-1152-485a-abff-1e8a04657771                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 91ce64af-10ec-4692-8c55-9accb39f7d2d
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 91ce64af-10ec-4692-8c55-9accb39f7d2d
       Status: Executing Task...
    └── 🤖 Agent: Business Analyst
            Status: In Progress

# Agent: Business Analyst
## Task: ยอดขายผลิตภัณฑ์ Xtralife เปลี่ยนแปลงไปอย่างไรบ้างในช่วงปีที่ผ่านมา.


🚀 Crew: crew
└── 📋 Task: 91ce64af-10ec-4692-8c55-9accb39f7d2d
       Status: Executing Task...
    └── 🤖 Agent: Business Analyst
            Status: In Progress
        └── 🧠 Thinking...

...

🚀 Crew: crew
└── 📋 Task: 91ce64af-10ec-4692-8c55-9accb39f7d2d
       Status: Executing Task...
    ├── 🤖 Agent: Business Analyst
    │       Status: In Progress
    │   └── 🔧 Using Ask question to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (4)
    │   └── 🔧 Using Delegate work to coworker (1)
    └── 🤖 Agent: Business Analyst
            Status: ✅ Completed
        └── 🔧 Used NL2SQLTool (8)

🚀 Crew: crew
└── 📋 Task: 91ce64af-10ec-4692-8c55-9accb39f7d2d
       Assigned to: Business Analyst
       Status: ✅ Completed
    ├── 🤖 Agent: Business Analyst
    │       Status: In Progress
    │   └── 🔧 Using Ask question to coworker (1)
    ├── 🤖 Agent: Database Analyst
    │       Status: ✅ Completed
    │   ├── 🔧 Used NL2SQLTool (4)
    │   └── 🔧 Using Delegate work to coworker (1)
    └── 🤖 Agent: Business Analyst
            Status: ✅ Completed
        └── 🔧 Used NL2SQLTool (8)

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 91ce64af-10ec-4692-8c55-9accb39f7d2d                                                                     │
│  Agent: Business Analyst                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6a004c8c-1152-485a-abff-1e8a04657771                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Raw Output: ยอดขายผลิตภัณฑ์ Xtralife ในปีที่ผ่านมา สามารถคำนวณได้โดยใช้คำถาม SQL ต่อไปนี้
```sql
SELECT 
  sales."OrderDate", 
  sales."Sales"
FROM 
  sales
WHERE 
  sales."ProductName" = 'Xtralife' 
  AND sales."OrderDate" BETWEEN '2022-01-01' AND '2022-12-31'
```
การวิเคราะห์ข้อมูลการขายผลิตภัณฑ์ Xtralife ในปีที่ผ่านมา จะทำให้เราทราบถึงการเปลี่ยนแปลงของยอดขายในแต่ละเดือนและช่วยให้เราสามารถวางแผนกลยุทธ์การขายได้อย่างมีประสิทธิภาพในอนาคต 

ตัวอย่างผลลัพธ์ของคำถาม SQL:
| OrderDate | Sales |
| --- | --- |
| 2022-01-01 | 100 |
| 2022-01-02 | 120 |
| ... | ... |
| 2022-12-30 | 150 |
| 2022-12-31 | 180 |

ผลลัพธ์นี้แสดงถึงข้อมูลการขายของผลิตภัณฑ์ Xtralife ในปี 2022 โดยไม่มีการคำนวณจำนวนการขายรายเดือนและ มูลค่าการขาย 

การคำนวณจำนวนการขายรายเดือนและ มูลค่าการขาย สามารถทำได้โดยใช้ฟังก์ชัน SUM และ GROUP BY ใน SQL 
```sql
SELECT 
  TO_CHAR(sales."OrderDate", 'MM') as "Month", 
  SUM(sales."Sales") as "TotalSales"
FROM 
  sales
WHERE 
  sales."ProductName" = 'Xtralife' 
  AND sales."OrderDate" B